### Prepare environment

In [0]:
%run ../environment/prepare_environment

# Autoencoders - Unsupervised Anomaly Detection

## Introduction

Autoencoders are unsupervised neural networks designed to compress input data into a lower-dimensional "latent space" and then reconstruct it back to its original form. By forcing the data through this narrow bottleneck, the model learns to ignore noise and capture only the most essential structural patterns. 

In this notebook, we will train the model exclusively on "round" digits (0, 3, 6, 8, 9) so it becomes an expert at reconstructing curved shapes. When presented with "straight" digits (1, 2, 4, 5, 7), the model fails to reconstruct them accurately, resulting in a significantly higher reconstruction error that allows us to flag them as anomalies.

This notebook will cover:
- Understanding autoencoders for unsupervised representation learning
- Using reconstruction error as an anomaly detector
- Training on 'normal' data to identify outliers
- Applications in fraud detection and cybersecurity
- Interpreting reconstruction difference heatmaps

## 1. Load MNIST and filter

We filter the MNIST dataset to include only "round" digits (0, 3, 6, 8, 9), defining these curved patterns as our normal baseline. The autoencoder learns to compress and reconstruct these specific geometries through its bottleneck, while any straight-edged digits (1, 2, 4, 5, 7) are treated as anomalies.

This follows the one-class classification paradigm: the model becomes an expert on the "normal" distribution and flags anything it cannot reconstruct accurately as an outlier. In real-world applications, this "normal" state could represent legitimate financial transactions, healthy medical scans, or stable network traffic, where any significant deviation triggers an alert.

In [0]:
import os
import random
import numpy as np
import tensorflow as tf
import mlflow
import mlflow.tensorflow
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from tensorflow.keras.callbacks import EarlyStopping

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

seed = 10
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

normal_digits = [0, 3, 6, 8, 9]
anomaly_digits = [1, 2, 4, 5, 7]

train_mask = np.isin(y_train, normal_digits)
x_train_normal = x_train[train_mask]

test_mask_normal = np.isin(y_test, normal_digits)
test_mask_anomaly = np.isin(y_test, anomaly_digits)

x_test_normal = x_test[test_mask_normal]
x_test_anomaly = x_test[test_mask_anomaly]

print(f"Max value: {x_train_normal.max()}") # Powinno być 1.0
print(f"Shape: {x_train_normal.shape}")

## 2. Build autoencoder

Unlike standard MLPs, this model uses spatial intelligence to distinguish between "curved" and "straight" geometries:
- **Encoder (Conv2D + MaxPooling)**: Filters scan the image to capture local spatial patterns like curves and loops. Successive pooling layers reduce the dimensions (28×28→14×14→7×7), distilling the image into high-level features.
- **Bottleneck (Dense Latent Space)**: The spatial data is flattened and compressed into a tiny 10-dimensional vector. By destroying the spatial grid and forcing the data through this "needle's eye," the model is prevented from simply copying pixels; it must instead learn an abstract mathematical "concept" of what a rounded digit looks like.
- **Decoder (Dense + Conv2DTranspose)**: The decoder first expands the 10-digit code back into a 7×7 grid, then uses Batch Normalization and UpSampling to rebuild the image. This stage is trained to "hallucinate" the missing details based on the circular patterns it learned during training.
- **Output Activation (Sigmoid)**: Constrains the output to [0,1], allowing us to treat pixels as probabilities. We use Binary Crossentropy (BCE) to compare the reconstruction to the original, as it is much more sensitive to structural mismatches than MSE, making anomalies easier to spot.

The Mechanism of Anomaly Detection
- **Reconstruction Failure**: Because the model is trained only on "round" shapes, it lacks the "vocabulary" to describe straight lines.
- **Anomalous Signal**: When a digit like 1 or 7 passes through, the decoder fails to recreate the sharp angles, resulting in a blurry, distorted output and a high MSE.
- **Thresholding**: Any image with an error exceeding a set limit (e.g., Mean+2×Std Dev) is automatically flagged as an anomaly.

The bottleneck forces abstraction: the model must learn what's 'essential' about the data to recreate it.

We minimize **Mean Squared Error (MSE)** between input and reconstruction. For "normal" digits, the model learns the patterns. For unseen digits, reconstruction error is high because the model doesn't know how to represent them efficiently.

This is the core insight for anomaly detection: normal data has low error, anomalies have high error. A threshold will separate the two.

In [0]:
autoencoder = keras.Sequential([
    # Encoder
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)), # 14x14
    layers.Conv2D(8, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)), # 7x7

    # Bottleneck
    layers.Flatten(),
    layers.Dense(10, activation='relu', name="bottleneck"), 
    
    #Decoder
    layers.Dense(7 * 7 * 8, activation='relu'),
    layers.Reshape((7, 7, 8)),
    layers.Conv2DTranspose(8, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.UpSampling2D((2, 2)), # 14x14
    layers.Conv2DTranspose(16, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.UpSampling2D((2, 2)), # 28x28
    layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')
])

optimizer = optimizers.Adam(learning_rate=0.01)
autoencoder.compile(optimizer=optimizer, loss='binary_crossentropy')

autoencoder.summary()

## 3. Train on normal data

Training the autoencoder is unsupervised: no labels are required. The model learns a "self-mapping" by comparing the original input to its own reconstructed output.

**Loss Function (Binary Crossentropy)**:
Instead of simple pixel distance, we use Binary Crossentropy (BCE). It measures the difference between the probability distributions of the input and the output pixels. For normal, "rounded" digits, BCE converges to low values as the model masters the geometric patterns. The bottleneck forces efficient encoding, compelling the model to discard noise and retain only the essential structural features of the trained digits.

**Validation Monitoring**:
A Validation Split (20%) monitors reconstruction quality on held-out normal digits. If validation loss begins to diverge from training loss, it indicates the model is memorizing (overfitting) specific images rather than learning the general rules of circular shapes.

**Post-Training Detection Logic**:
- Normal Digits: High reconstruction quality, resulting in Low BCE/MSE.
- Anomaly Digits: Poor, distorted reconstruction, resulting in Higher BCE/MSE.

This mathematical "confusion" is exactly what we exploit for anomaly detection. By establishing a threshold, we can automatically flag any digit that the model fails to reconstruct accurately.

In [0]:
mlflow.tensorflow.autolog(
    silent=True,
    registered_model_name='ai_ml_in_practice.neural_networks_silver.mnist_autoencoder_anomaly'
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

with mlflow.start_run(run_name='autoencoder_anomaly_detection') as run:
    # Notice: x_train_normal is used for BOTH input and target
    history = autoencoder.fit(
        x_train_normal, 
        x_train_normal, 
        epochs=10,
        batch_size=32, 
        validation_split=0.2, 
        callbacks=[early_stop],
        verbose=2
    )
    
    # Evaluate on the normal test set
    test_loss = autoencoder.evaluate(x_test_normal, x_test_normal, verbose=0)
    
    # Log the reconstruction error
    mlflow.log_metrics({
        "test_reconstruction_loss": float(test_loss)
    })

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(history.history['loss'], label='Train Loss')
    ax.plot(history.history['val_loss'], label='Val Loss')
    ax.set_title('Autoencoder Reconstruction Loss')
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Binary Crossentropy')
    ax.legend()
    
    mlflow.log_figure(fig, "autoencoder_training_summary.png")
    plt.show()

print(f"Run complete. Model registered as 'mnist_autoencoder_anomaly'.")

## 4. Anomaly demo

The Difference Heatmap visualizes pixel-level reconstruction failure. Because the model was forced through a 10-dimensional bottleneck, it can only reconstruct shapes it has "conceptualized" during training.

Why do anomalies (1, 2, 4, 5, 7) trigger higher errors?
- **Feature Mismatch**: The encoder has learned to compress loops and arcs. When it sees the sharp angles of a 7 or the straight line of a 1, it lacks the mathematical vocabulary to represent them.
- **Hallucination**: The decoder attempts to "fix" the anomaly by forced reconstruction using learned patterns. It often tries to turn a 7 into a blurry 9 or 0, creating massive pixel-wise differences.
- **BCE Sensitivity**: Using Binary Crossentropy ensures that even small structural deviations (like a missing corner) result in a significant penalty compared to standard MSE.

**Setting the Threshold**: We typically set a threshold based on mean and standard deviation.
- Below Threshold → Normal: The data conforms to the learned "curved" distribution.
- Above Threshold → Anomaly: The data is structurally inconsistent with the training set.

**Real-World Applications**: This "train on normal, flag the weird" logic is the industry standard for scenarios where you have plenty of "good" data but very few examples of "bad" data.
- **Banking & Fraud**: Train on millions of legitimate transactions. A sudden change in spending patterns or location triggers a high reconstruction error, flagging potential fraud.
- **Industrial IoT**: Train on sensor data from a healthy turbine. When a bearing starts to wear out, the vibrations become "anomalous" to the model, predicting failure before it happens.
- **Cybersecurity**: Train on standard network traffic. A DDoS attack or an unauthorized data exfiltration attempt creates a pattern the model cannot efficiently reconstruct, triggering an alert.

In [0]:
loaded_model = mlflow.tensorflow.load_model(
    "models:/ai_ml_in_practice.neural_networks_silver.mnist_autoencoder_anomaly/1"
)

In [0]:
target_digit = 5

digit_indices = np.where(y_test == target_digit)[0]
idx = random.choice(digit_indices)
x_digit = x_test[idx:idx+1]

x_rec = loaded_model.predict(x_digit, verbose=0)

diff = np.abs(x_digit - x_rec).reshape(28, 28)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title(f'Original Digit: {target_digit}')
plt.imshow(x_digit.reshape(28, 28), cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title(f'Reconstructed {target_digit}')
plt.imshow(x_rec.reshape(28, 28), cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title('Difference Heatmap (Error)')
plt.imshow(diff, cmap='hot')
plt.colorbar(label='Pixel Difference')
plt.axis('off')

plt.tight_layout()
plt.show()

reconstructed_all = loaded_model.predict(x_test, verbose=0)
mse_scores = np.mean(np.square(x_test - reconstructed_all), axis=(1, 2, 3))

print(f'--- Error Statistics for target_digit {target_digit} ---')
print(f'Mean MSE for all test data: {np.mean(mse_scores):.4f}')
print(f'Max MSE found in test set:  {np.max(mse_scores):.4f}')

sample_mse = np.mean(np.square(x_digit - x_rec))
print(f'MSE for this specific sample: {sample_mse:.4f}')